In [2]:
import pandas as pd
import numpy as np

In [3]:
import os

In [2]:
#!pip install python-docx

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.1 MB 4.8 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.1 MB 5.1 MB/s eta 0:00:01
   ------------------------------ --------- 3.1/4.1 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 5.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   ---------------------------------------- 2/2 [python-docx]



In [4]:
import docx

In [5]:
resume_folder = r"C:\Users\91904\OneDrive\Documents\Resumes"
def read_docx(file_path):
    doc = docx.Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text

all_resumes = []

for file in os.listdir(resume_folder):
    if file.endswith(".docx"):
        text = read_docx(os.path.join(resume_folder, file))
        all_resumes.append({"filename": file, "text": text})

print("Total resumes loaded:", len(all_resumes))
print("First resume sample:\n", all_resumes[0]["text"][:300])

Total resumes loaded: 8
First resume sample:
 
Amit Choudhary
Supply Chain / Operations
Kolkata, West Bengal | +91 98XXXXXX17 | amit.choudhary@email.com
PROFESSIONAL SUMMARY
Operations professional with 5 years in logistics, inventory and vendor management for FMCG distribution.
KEY SKILLS
Inventory control, demand planning, SAP MM/SD, vendor n


In [6]:
# for i in range(3):  # show first 3 resumes
#     print("Resume:", all_resumes[i]["filename"])
#     print(all_resumes[i]["text"][:300], "\n")

In [6]:
from ollama import chat
import json
import time
import re

def clean_json(text):
    # Extract JSON substring between { ... }
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return None

batch_size = 2
for i in range(0, len(all_resumes), batch_size):
    batch = all_resumes[i:i+batch_size]
    print(f"\n🔄 Processing batch {i//batch_size + 1} ({len(batch)} resumes)...")

    for resume in batch:

        json_file = resume['filename'].replace(".docx", ".json")
        if os.path.exists(json_file) and os.path.getsize(json_file) > 0:
            print(f"⏭️ Skipped {json_file} (already converted)")
            continue
            
        prompt = f"""
        Extract the following information from the resume:

        - Name
        - Job Title
        - Phone
        - Email
        - Location
        - Skills
        - Education
        - Experience

        Return ONLY valid JSON.
        Do not add explanations or markdown.

        Resume:
        {resume['text']}
        """

        response = chat(
            model="llama3.2:latest",
            messages=[{"role": "user", "content": prompt}]
        )
        parsed_resume = response["message"]["content"]

         # ✅ Clean JSON before parsing
        parsed_json_str = clean_json(parsed_resume)

        if parsed_json_str:
            try:
                parsed_json = json.loads(parsed_json_str)

                # Save nicely formatted JSON
                with open(json_file, "w") as f:
                    json.dump(parsed_json, f, indent=4)

                print(f"✅ Saved {json_file}")
            except json.JSONDecodeError:
                print(f"❌ Skipped {resume['filename']} (invalid JSON after cleaning)")
        else:
            print(f"❌ Skipped {resume['filename']} (no JSON found)")

        # ⏸ Small pause to reduce load
        time.sleep(10)


🔄 Processing batch 1 (2 resumes)...
⏭️ Skipped Amit Choudhary.json (already converted)
⏭️ Skipped Ananya Iyer.json (already converted)

🔄 Processing batch 2 (2 resumes)...
⏭️ Skipped Anitha Joseph.json (already converted)
⏭️ Skipped Arjun Reddy.json (already converted)

🔄 Processing batch 3 (2 resumes)...
⏭️ Skipped Deepa Menon.json (already converted)
⏭️ Skipped Farhan Sheikh.json (already converted)

🔄 Processing batch 4 (2 resumes)...
⏭️ Skipped Ishita Bose.json (already converted)
⏭️ Skipped Karthik Subramanian.json (already converted)


In [7]:
import glob
# Load all JSON files in the folder
resumes = []
for file in glob.glob("*.json"):
    with open(file) as f:
        resumes.append(json.load(f))

print(f"Loaded {len(resumes)} resumes")


Loaded 8 resumes


In [8]:
df = pd.DataFrame(resumes)
print(df.head())

             name           phone                     email  \
0  Amit Choudhary  +91 98XXXXXX17  amit.choudhary@email.com   
1     Ananya Iyer  +91 98XXXXXX03     ananya.iyer@email.com   
2   Anitha Joseph  +91 98XXXXXX12   anitha.joseph@email.com   
3     Arjun Reddy  +91 98XXXXXX02     arjun.reddy@email.com   
4     Deepa Menon  +91 98XXXXXX14     deepa.menon@email.com   

                     location  \
0        Kolkata, West Bengal   
1        Bengaluru, Karnataka   
2  Thiruvananthapuram, Kerala   
3        Hyderabad, Telangana   
4        Bengaluru, Karnataka   

                                              skills  \
0  [Inventory control, demand planning, SAP MM/SD...   
1  [Python, SQL, Pandas, Scikit-learn, TensorFlow...   
2  [ICU care, ventilator management, IV therapy, ...   
3  Java, Spring Boot, React, REST APIs, MySQL, Mo...   
4  ["Agile/Scrum", "Jira", "risk management", "st...   

                                           education  \
0                         [PG

In [9]:
df.to_csv("resumes_dataset.csv", index=False)

In [10]:
print(os.getcwd())

C:\Users\91904\Documents\Python Programming\Jupyter\JobFit-AI Resume Analyzer
